# Exploratory Data Analysis — Security Logs

This notebook explores the cybersecurity dataset before building the main application.
We use **Pandas** for data loading and analysis, **Numpy** for numerical calculations,
and **Matplotlib** for visualizations.

Dataset source: https://www.kaggle.com/datasets/aryan208/cybersecurity-threat-detection-logs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Load the dataset

In [ ]:
df = pd.read_csv("../data/logs.csv", parse_dates=["timestamp"])
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Threat label distribution

The dataset classifies each log entry as `benign`, `suspicious`, or `malicious`.

In [ ]:
label_counts = df["threat_label"].value_counts()
label_counts

In [ ]:
colors = ["#2ecc71", "#f39c12", "#e74c3c"]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    label_counts.values,
    labels=[lbl.capitalize() for lbl in label_counts.index],
    colors=colors,
    autopct="%1.1f%%",
    startangle=140,
)
ax.set_title("Threat Label Distribution")
plt.tight_layout()
plt.show()

## Bytes transferred — Numpy analysis

We convert the column to a Numpy array and calculate basic statistics.

In [ ]:
bytes_arr = df["bytes_transferred"].to_numpy()

print(f"Mean:   {np.mean(bytes_arr):,.0f} B")
print(f"Median: {np.median(bytes_arr):,.0f} B")
print(f"Std:    {np.std(bytes_arr):,.0f} B")
print(f"Min:    {np.min(bytes_arr):,} B")
print(f"Max:    {np.max(bytes_arr):,} B")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(bytes_arr, bins=50, color="#3498db", edgecolor="white")
ax.set_xlabel("Bytes Transferred")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Bytes Transferred per Session")
plt.tight_layout()
plt.show()

## Top source IPs

Which IP addresses appear most frequently in the logs?

In [ ]:
top_ips = df["source_ip"].value_counts().head(10)
top_ips

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_ips.index[::-1], top_ips.values[::-1], color="#3498db")
ax.set_xlabel("Connection Count")
ax.set_title("Top 10 Source IPs by Connection Frequency")
plt.tight_layout()
plt.show()

## Activity over time

How does network activity vary across the year? We use `resample()` to aggregate by day.

In [ ]:
daily = df.set_index("timestamp").resample("D").size()
daily.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily.values, color="#3498db", linewidth=1.5)
ax.fill_between(daily.index, daily.values, alpha=0.15, color="#3498db")
ax.set_xlabel("Date")
ax.set_ylabel("Log Entries")
ax.set_title("Daily Network Activity")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Bytes transferred per threat label

Does malicious traffic use more bandwidth than benign traffic?

In [ ]:
grouped = df.groupby("threat_label")["bytes_transferred"].agg(["mean", "max", "sum"])
grouped

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(grouped.index, grouped["mean"], color=["#2ecc71", "#e74c3c", "#f39c12"])
ax.set_xlabel("Threat Label")
ax.set_ylabel("Mean Bytes Transferred")
ax.set_title("Average Bytes Transferred by Threat Label")
plt.tight_layout()
plt.show()